# Notebook Instructions

1. If you are new to Jupyter notebooks, please go through this introductory manual <a href='https://quantra.quantinsti.com/quantra-notebook' target="_blank">here</a>.
1. Any changes made in this notebook would be lost after you close the browser window. **You can download the notebook to save your work on your PC.**
1. Before running this notebook on your local PC:<br>
i.  You need to set up a Python environment and the relevant packages on your local PC. To do so, go through the section on "**Run Codes Locally on Your Machine**" in the course.<br>
ii. You need to **download the zip file available in the last unit** of this course. The zip file contains the data files and/or python modules that might be required to run this notebook.

# Random Forest
Random Forest, also called Random Decision Forest, is a method in machine learning capable of performing both regression and classification tasks. It is a type of ensemble learning that uses multiple learning algorithms for prediction.

Random Forest comprises of decision trees, which are graphs of decisions representing their course of action or statistical probability. These multiple trees are plotted to a single tree called the Classification and Regression (CART) Model. To classify an object based on its attributes, each tree gives a classification that is said to vote for that class. The forest then chooses the classification with the maximum number of votes. For regression, it considers the average of the outputs for different trees.

<b>Working</b>
1. It assumes the number of cases as N. Then, randomly but with replacement, the sample of these N cases is taken out, which will be the training set.
2. Considering M to be the input variables, a number m is selected such that m < M. The best split between m and M is used to split the node. The value of m is held constant as the trees are grown.
3. Each tree is grown as large as possible.
4. By aggregating the predictions of n trees (i.e., majority votes for classification, the average for
regression), random forest predicts the new data.


![alt text](https://d2a032ejo53cab.cloudfront.net/Glossary/UclBTDKs/image-3.png)

For example, in the above diagram, we can observe that each decision tree has voted or predicted a specific class. The final output or class selected by the Random Forest will be the Class N, as it has majority votes or is the predicted output by two out of the four decision trees.



Random Forest has certain advantages and disadvantages.

<b>Advantages</b>
1. This method balances the errors which are present in the dataset.
2. It is an effective method because it maintains accuracy even if it has to estimate the missing data.
3. Using the out-of-bag error estimate removes the need for a set-aside test set.
4. Random Forest helps in unsupervised clustering, data views, and outlier detection.
5. It reduces data management time and pre-processing tasks. 

<b>Disadvantages</b>

Disadvantages of the random forest may include its inability to be at par excellence for the regression problem as it does not give precise continuous nature predictions. It cannot predict beyond the range in the
training data. Further, it does not provide complete control to the modeller.

<b>Applications of Random Forest</b>
1. It has many application in computational biology. Doctors can estimate the drug response to a particlar disease using this model.
2. This can be used to calculate a person's credit rating by comparing with other persons having similar traits.
3. 



You can learn more about Random Forest and their application in trading in <a href="https://blog.quantinsti.com/random-forest-algorithm-in-python/"> this article</a>. 

In this notebook, you will perform the following steps:

1. [Import Data](#data)


2. [Independent Variables](#x)


3. [Dependent Variable](#y)


4. [Split the Dataset](#split)


5. [Train the Model](#model)


6. [Accuracy Score](#score)   

## Import library

In [1]:
# For data manipulation
import numpy as np
import pandas as pd

# Import RandomForestClassifier and accuracy_score functions from sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

import warnings
warnings.filterwarnings("ignore")

<a id='data'></a> 

## Import Data

We will read the daily data of stock, Bank of America, to create features.

In [2]:
# The data is stored in the directory 'data'
path = '../data/'

# Read stock data from csv file
df = pd.read_csv(path + 'SPY.csv', index_col=0)
df.index = pd.to_datetime(df.index)
df.head()

,Open,High,Low,Close,Volume
Date,,,,,
2018-12-31,2498.939941,2509.239990,2482.820068,2506.850098,3442870000
2019-01-02,2476.959961,2519.489990,2467.469971,2510.030029,3733160000
2019-01-03,2491.919922,2493.139893,2443.959961,2447.889893,3822860000
2019-01-04,2474.330078,2538.070068,2474.330078,2531.939941,4213410000
2019-01-07,2535.610107,2566.159912,2524.560059,2549.689941,4104710000


In [3]:
df.tail()

,Open,High,Low,Close,Volume
Date,,,,,
2020-07-28,3234.270020,3243.719971,3216.169922,3218.439941,4027890000
2020-07-29,3227.219971,3264.739990,3227.219971,3258.439941,4676300000
2020-07-30,3231.760010,3250.919922,3204.129883,3246.219971,4254010000
2020-07-31,3270.449951,3272.169922,3220.260010,3271.120117,5117260000
2020-08-03,3288.260010,3302.729980,3284.530029,3294.610107,4643640000


<a id='x'></a> 

## Independent Variables
We will create independent variables which consist of 4 features. The features are:
1. Ratio of open and close price
2. Ratio of high and low price
3. 1-day lag returns
4. 2-day lag returns

In [4]:
# Create input features

df['return'] = df['Close'].pct_change()
df['Open/Close'] = (df['Open'] / df['Close'])
df['High/Low'] = (df['High'] / df['Low'])
df['1_day_lag_returns'] = df['return'].shift(1)
df['2_day_lag_returns'] = df['return'].shift(2)

# Drop NaN values
df.dropna(inplace=True)

# Store the features in a variable X
X = df[['Open/Close', 'High/Low','1_day_lag_returns','2_day_lag_returns']]
X.head()

,Open/Close,High/Low,1_day_lag_returns,2_day_lag_returns
Date,,,,
2019-01-04,0.977247,1.025761,-0.024757,0.001268
2019-01-07,0.994478,1.016478,0.034336,-0.024757
2019-01-08,0.997553,1.012663,0.007010,0.034336
2019-01-09,0.998081,1.010289,0.009695,0.007010
2019-01-10,0.991092,1.013973,0.004098,0.009695


<a id='y'></a> 

## Dependent Variable 
When the next day's close price is greater than today's close price, we use 1 as a signal and else use -1. We will store this in the variable y, which is the dependent/target variable.

In [5]:
y = np.where(df['Close'].shift(-1) > df['Close'], 1, -1)
y[:10]

array([ 1,  1,  1,  1, -1, -1,  1,  1,  1,  1])

<a id='split'></a> 

## Split the Dataset
We will split the dataset into train and test samples. The train data consists of 75% of the total datasets. On the remaining, we will test the accuracy of the model.

In [6]:
# Training dataset length
split = int(len(df) * 0.75)

# Splitting the X and y into train and test datasets
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

<a id='model'></a> 

## Train the Model

We will use the `RandomForestClassifier` function from sklearn to train and fit the model. The syntax of the followinf function is as follows:
```python
RandomForestClassifier()
```
Parameters: 

    1. random_state (Make it any constant value to produce same results)
    
    
Returns: 

    1. Predicting the test dataset
 

In [7]:
# Create and fit the model on train dataset
clf = RandomForestClassifier(random_state=5)
model = clf.fit(X_train, y_train)

<a id='score'></a> 

## Accuracy Score
The model is trained on the training dataset. Now it's time to test the accuracy of the model on the test dataset. We will use `accuracy_score` function to test the accuracy.The syntax of the followinf function is as follows:
```python
accuracy_score()
```
Parameters: 

    1. y_test
    2. Predicted y_test
    2. Normalization parameter
    
    
Returns: 

    1. Prediction accuracy
 

In [8]:
print('Prediction Accuracy (%): ', round(accuracy_score(y_test, model.predict(X_test), normalize=True)*100.0,2))

Prediction Accuracy (%):  59.0


This is a very simple model with an accuracy of around 57.58% on the test dataset.

## Conclusion
In this notebook, we learnt the functioning of the Random Forest Algorithm with the help of an example, along with the Python code to implement this strategy.
<br><br>  